In [2]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

# ==============================
#  CONFIGURATION
# ==============================
INPUT_DIR = "original"             # folder with 3 images (normal, dark, bright)
OUTPUT_DIR = "enhanced_results"    # folder to store enhanced images
COMPARE_DIR = "comparison"         # folder to store comparison collages
SHOW_COMPARISON = False            # change to True to display collages live

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(COMPARE_DIR, exist_ok=True)

# ==============================
#  CONTRAST ENHANCEMENT FUNCTIONS
# ==============================
def linear_enhancement(img):
    img = img.astype(np.float32) / 255.0
    a, b = 1.2, 0.05  # tuneable parameters
    enhanced = np.clip(a * img + b, 0, 1)
    return (enhanced * 255).astype(np.uint8)

def log_enhancement(img):
    img = img.astype(np.float32) / 255.0
    c = 1.0 / np.log(1 + np.max(img))
    enhanced = c * np.log(1 + img)
    enhanced = np.clip(enhanced, 0, 1)
    return (enhanced * 255).astype(np.uint8)

def exp_enhancement(img):
    img = img.astype(np.float32) / 255.0
    c = np.exp(img) - 1
    c = c / np.max(c)
    enhanced = np.clip(c, 0, 1)
    return (enhanced * 255).astype(np.uint8)

# ==============================
#  RGB ↔ HSI CONVERSION
# ==============================
def rgb_to_hsi(rgb):
    rgb = rgb.astype(np.float32) / 255.0
    R, G, B = rgb[:,:,0], rgb[:,:,1], rgb[:,:,2]
    num = 0.5 * ((R - G) + (R - B))
    den = np.sqrt((R - G)**2 + (R - B)*(G - B)) + 1e-6
    theta = np.arccos(np.clip(num / den, -1, 1))

    H = np.where(B <= G, theta, 2*np.pi - theta)
    H = H / (2*np.pi)

    min_rgb = np.minimum(np.minimum(R, G), B)
    S = 1 - (3 / (R + G + B + 1e-6)) * min_rgb
    I = (R + G + B) / 3

    return np.stack([H, S, I], axis=-1)

def hsi_to_rgb(hsi):
    H, S, I = hsi[:,:,0]*2*np.pi, hsi[:,:,1], hsi[:,:,2]
    R, G, B = np.zeros_like(H), np.zeros_like(H), np.zeros_like(H)

    # Sector 0 to 120 degrees
    idx = (H < 2*np.pi/3)
    B[idx] = I[idx]*(1 - S[idx])
    R[idx] = I[idx]*(1 + (S[idx]*np.cos(H[idx]))/(np.cos(np.pi/3 - H[idx])))
    G[idx] = 3*I[idx] - (R[idx] + B[idx])

    # Sector 120 to 240 degrees
    idx = (H >= 2*np.pi/3) & (H < 4*np.pi/3)
    H2 = H[idx] - 2*np.pi/3
    R[idx] = I[idx]*(1 - S[idx])
    G[idx] = I[idx]*(1 + (S[idx]*np.cos(H2))/(np.cos(np.pi/3 - H2)))
    B[idx] = 3*I[idx] - (R[idx] + G[idx])

    # Sector 240 to 360 degrees
    idx = (H >= 4*np.pi/3)
    H3 = H[idx] - 4*np.pi/3
    G[idx] = I[idx]*(1 - S[idx])
    B[idx] = I[idx]*(1 + (S[idx]*np.cos(H3))/(np.cos(np.pi/3 - H3)))
    R[idx] = 3*I[idx] - (G[idx] + B[idx])

    rgb = np.clip(np.stack([R, G, B], axis=-1), 0, 1)
    return (rgb * 255).astype(np.uint8)

# ==============================
#  PROCESS EACH IMAGE
# ==============================
def process_image(image_path, img_name):
    print(f"Processing {img_name} ...")
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # --- Define enhancement methods ---
    enhancements = {
        'Linear': linear_enhancement,
        'Log': log_enhancement,
        'Exp': exp_enhancement
    }

    results = {'Original': img}

    # --- Apply each enhancement method ---
    for method_name, func in enhancements.items():
        # --- RGB enhancement ---
        r, g, b = cv2.split(img)
        rE, gE, bE = func(r), func(g), func(b)
        rgb_combined = cv2.merge((rE, gE, bE))
        results[f'RGB_{method_name}'] = rgb_combined

        # --- HSI enhancement ---
        hsi = rgb_to_hsi(img)
        for comp_idx, comp_name in enumerate(['H', 'S', 'I']):
            hsi_mod = hsi.copy()
            comp = (hsi[:,:,comp_idx] * 255).astype(np.uint8)
            compE = func(comp)
            hsi_mod[:,:,comp_idx] = compE.astype(np.float32)/255.0
            results[f'{comp_name}_{method_name}'] = hsi_to_rgb(hsi_mod)

    # --- Save all results individually ---
    for name, out_img in results.items():
        save_path = os.path.join(OUTPUT_DIR, f"{img_name}_{name}.png")
        cv2.imwrite(save_path, cv2.cvtColor(out_img, cv2.COLOR_RGB2BGR))

    # --- Create comparison collage ---
    fig, axes = plt.subplots(4, 4, figsize=(14, 14))
    fig.suptitle(f"Comparison for {img_name}", fontsize=16)

    all_keys = list(results.keys())  # total 13 images
    for i, ax in enumerate(axes.flat):
        if i < len(all_keys):
            ax.imshow(results[all_keys[i]])
            ax.set_title(all_keys[i], fontsize=10)
        else:
            ax.axis('off')
        ax.axis('off')

    plt.tight_layout()
    plt.subplots_adjust(top=0.93)
    comparison_path = os.path.join(COMPARE_DIR, f"{img_name}_comparison.png")
    plt.savefig(comparison_path)

    if SHOW_COMPARISON:
        plt.show()
    plt.close()
    print(f"✅ Saved comparison collage for {img_name} in {COMPARE_DIR}")

# ==============================
#  MAIN EXECUTION
# ==============================
for file in os.listdir(INPUT_DIR):
    if file.lower().endswith(('.png', '.jpg', '.jpeg')):
        name = os.path.splitext(file)[0]
        process_image(os.path.join(INPUT_DIR, file), name)

print("\n🎉 All images processed and saved successfully in:")
print("   Enhanced images →", OUTPUT_DIR)
print("   Comparison collages →", COMPARE_DIR)


Processing bright ...
✅ Saved comparison collage for bright in comparison
Processing dark ...
✅ Saved comparison collage for dark in comparison
Processing normal ...
✅ Saved comparison collage for normal in comparison

🎉 All images processed and saved successfully in:
   Enhanced images → enhanced_results
   Comparison collages → comparison
